# Predicción con Series Temporales Históricas

**Objetivo**: entender que hace especial a una serie temporal, como convertirla
en un problema supervisado mediante ventana deslizante, y cuando usar estadística
vs ML vs DL.

**Conexion con el documento**: seccion 5 - Arquitecturas Profundas.

In [ ]:

# cell-00-setup-entorno
import os, sys
from pathlib import Path

# Este notebook genera sus propios datos sinteticos (data/churn_series.csv)
# Solo necesitamos asegurarnos de que el CWD sea el directorio del notebook
if "__vsc_ipynb_file__" in dir():
    os.chdir(Path(__vsc_ipynb_file__).parent)
elif not Path("data").exists() and not Path("images").exists():
    for _p in [Path("Jupyter_notebooks"), Path("../Jupyter_notebooks")]:
        if _p.exists():
            os.chdir(_p)
            break

print(f"Entorno listo. CWD: {os.getcwd()}")


In [1]:
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
IMG = pathlib.Path('images')
IMG.mkdir(exist_ok=True)
DATA = pathlib.Path('data')
DATA.mkdir(exist_ok=True)
print('Setup OK')

Setup OK


---

## 1. Que hace especial a una serie temporal?

En un dataset tabular clásico, el orden de las filas no importa. Puedes barajar
los registros y el modelo aprende lo mismo. Una serie temporal rompe esta suposicion:

> **El orden ES información.** Si barajas los meses, destruyes toda la estructura predictiva.

La diferencia fundamental con datos tabulares es que aqui el tiempo crea **dependencias
hacia atras**: el churn de enero influye en el de febrero, que influye en el de marzo.
Esta dependencia temporal es la razon por la que no puedes usar un split aleatorio
para train/test - si lo haces, estaras entrenando con datos del futuro.

Las tres propiedades que distinguen una serie temporal:
- **Autocorrelacion**: el valor actual depende de valores pasados
- **Tendencia**: dirección sistemática a largo plazo (subida, bajada)
- **Estacionalidad**: patrón que se repite en ciclos regulares (anual, mensual)

In [2]:
import numpy as np
import pandas as pd
import pathlib

np.random.seed(42)
DATA = pathlib.Path('data')
DATA.mkdir(exist_ok=True)

meses = pd.date_range('2020-01', periods=48, freq='MS')
mes_str = [m.strftime('%Y-%m') for m in meses]

# Tendencia: decreciente lineal de 0.12 a 0.06
trend = np.linspace(0.12, 0.06, 48)

# Estacionalidad: +0.02 en meses de verano (jun-sep), -0.01 en invierno
seasonality = np.zeros(48)
for i, m in enumerate(meses):
    if m.month in [6, 7, 8, 9]:
        seasonality[i] = 0.02
    elif m.month in [12, 1, 2]:
        seasonality[i] = -0.01

# Ruido gaussiano
noise = np.random.normal(0, 0.005, 48)

churn_rate = np.clip(trend + seasonality + noise, 0.01, 0.30)

df = pd.DataFrame({'mes': mes_str, 'churn_rate': churn_rate.round(5)})
df.to_csv('data/churn_series.csv', index=False)

print('Primeras 6 filas:')
print(df.head(6).to_string(index=False))
print()
print('Estadisticas basicas:')
print(df['churn_rate'].describe().round(5).to_string())
print()
print(f'Guardado: data/churn_series.csv  ({len(df)} filas)')

Primeras 6 filas:
    mes  churn_rate
2020-01     0.11248
2020-02     0.10803
2020-03     0.12069
2020-04     0.12379
2020-05     0.11372
2020-06     0.13245

Estadisticas basicas:
count    48.00000
mean      0.09314
std       0.02250
min       0.05297
25%       0.07659
50%       0.09288
75%       0.11007
max       0.14024

Guardado: data/churn_series.csv  (48 filas)


In [3]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('data/churn_series.csv')
serie = df['churn_rate'].values
x = np.arange(len(serie))

# Media movil de 3 meses como tendencia suavizada
rolling = pd.Series(serie).rolling(window=3, center=True).mean().values

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, serie * 100, color='#1565C0', linewidth=1.8,
        marker='o', markersize=4, label='Churn mensual')
ax.plot(x, rolling * 100, color='#E65100', linewidth=2.5,
        linestyle='--', label='Tendencia suavizada (media movil 3m)')

# Etiquetas de anio en el eje x
tick_pos  = [0, 12, 24, 36, 47]
tick_labs = ['Ene 2020', 'Ene 2021', 'Ene 2022', 'Ene 2023', 'Dic 2023']
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labs, fontsize=9)

ax.set_xlabel('Mes', fontsize=11)
ax.set_ylabel('Tasa de churn (%)', fontsize=11)
ax.set_title('Tasa de churn mensual la empresa (2020-2023)', fontsize=12, pad=10)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('images/B05A_fig01.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05A_fig01.png')

Guardado: images/B05A_fig01.png


> Antes de seguir: si tuvieras estos 48 meses de datos, que informacion usarias
> para predecir el mes 49? Cuantos meses anteriores son suficientes?

<details>
<summary>Orientacion para el instructor</summary>

Una respuesta madura menciona que no existe respuesta unica - depende de si hay
estacionalidad (entonces necesitas al menos 12 meses para capturar el ciclo completo)
o solo tendencia reciente (3-6 meses pueden bastar). La decision del tamaño de ventana
es una decision de negocio disfrazada de hiperparámetro técnico.

Senal de comprension: el alumno pregunta por el periodo de estacionalidad antes de
elegir la ventana. Si alguien dice 'uso todos los meses disponibles', preguntar:
cuanto pesas el churn de enero 2020 frente al de diciembre 2023 para predecir enero 2024?

</details>

In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = pd.read_csv('data/churn_series.csv')
serie = df['churn_rate'].values
n = len(serie)

# Tendencia: media movil de 12 meses (centrada)
trend = pd.Series(serie).rolling(window=12, center=True).mean().values

# Estacionalidad: promedio de (valor - tendencia) por mes del anio
detrended = serie - np.where(np.isnan(trend), np.nanmean(trend), trend)
meses_idx = [(i % 12) for i in range(n)]
seasonal_avg = np.zeros(12)
for m in range(12):
    idxs = [i for i in range(n) if meses_idx[i] == m and not np.isnan(trend[i])]
    if idxs:
        seasonal_avg[m] = np.mean(detrended[idxs])
seasonal = np.array([seasonal_avg[m] for m in meses_idx])

# Residuo
trend_fill = np.where(np.isnan(trend), np.nanmean(trend), trend)
residuo = serie - trend_fill - seasonal

x = np.arange(n)
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(x, trend_fill * 100, color='#1565C0', linewidth=2)
axes[0].set_ylabel('Tendencia (%)', fontsize=10)
axes[0].set_title('Tendencia (media movil 12m)', fontsize=10)
axes[0].grid(alpha=0.25)

axes[1].bar(x, seasonal * 100, color='#E65100', alpha=0.7, width=0.8)
axes[1].axhline(0, color='#333', linewidth=0.8)
axes[1].set_ylabel('Estacionalidad (%)', fontsize=10)
axes[1].set_title('Estacionalidad (patron anual promediado)', fontsize=10)
axes[1].grid(alpha=0.25)

axes[2].plot(x, residuo * 100, color='#2E7D32', linewidth=1.5,
             marker='o', markersize=3, alpha=0.8)
axes[2].axhline(0, color='#333', linewidth=0.8)
axes[2].set_ylabel('Residuo (%)', fontsize=10)
axes[2].set_title('Residuo (ruido no explicado)', fontsize=10)
axes[2].grid(alpha=0.25)

tick_pos  = [0, 12, 24, 36, 47]
tick_labs = ['Ene 2020', 'Ene 2021', 'Ene 2022', 'Ene 2023', 'Dic 2023']
axes[2].set_xticks(tick_pos)
axes[2].set_xticklabels(tick_labs, fontsize=9)

fig.suptitle('Descomposicion de la serie: tendencia + estacionalidad + residuo',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('images/B05A_fig02.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05A_fig02.png')

Guardado: images/B05A_fig02.png


---

## Ventana deslizante: convertir series en supervisado

La ventana deslizante (sliding window) transforma la serie en un dataset tabular:
- Input: churn de los ultimos N meses [t-N, ..., t-1]
- Output: churn del mes t

Esto convierte el problema en regresion clásica. El tamaño de la ventana es un
**hiperparámetro de negocio**: ventanas cortas capturan cambios recientes pero
pierden contexto estacional; ventanas largas incorporan mas contexto pero necesitan
mas datos y pueden incluir patrones que ya no son relevantes.

```
Serie original:  [0.10, 0.09, 0.08, 0.09, 0.07, 0.08, ...]

Con ventana N=3:
  X[0] = [0.10, 0.09, 0.08]   ->  y[0] = 0.09
  X[1] = [0.09, 0.08, 0.09]   ->  y[1] = 0.07
  X[2] = [0.08, 0.09, 0.07]   ->  y[2] = 0.08
  ...
```

El split temporal es obligatorio: los ultimos meses son siempre el test.

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('data/churn_series.csv')
serie = df['churn_rate'].values

# Ventana deslizante N=3
N = 3
X = np.array([serie[i:i+N] for i in range(len(serie)-N)])
y = serie[N:]

# Split 80/20 temporal (NO aleatorio)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Entrenar LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Ventana N={N} | MAE={mae:.5f} | RMSE={rmse:.5f}')
print()
print('Ultimos 6 meses - prediccion vs real:')
print(f'  {"Mes":<10} {"Real":>10} {"Predicho":>10} {"Error":>10}')
print('  ' + '-'*44)
meses_test = df['mes'].values[N+split:]
for i in range(-6, 0):
    m   = meses_test[i] if i + len(meses_test) >= 0 and abs(i) <= len(meses_test) else '---'
    r   = y_test[i]
    p   = y_pred[i]
    err = p - r
    print(f'  {m:<10} {r:>10.5f} {p:>10.5f} {err:>+10.5f}')

Ventana N=3 | MAE=0.00775 | RMSE=0.00916

Ultimos 6 meses -  prediccion vs real:
  Mes              Real   Predicho      Error
  --------------------------------------------
  2023-07       0.08580    0.08842   +0.00262
  2023-08       0.08360    0.09206   +0.00846
  2023-09       0.07644    0.08425   +0.00781
  2023-10       0.05895    0.07886   +0.01991
  2023-11       0.05897    0.06359   +0.00462
  2023-12       0.05529    0.05914   +0.00385


In [6]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df = pd.read_csv('data/churn_series.csv')
serie = df['churn_rate'].values

N = 3
X = np.array([serie[i:i+N] for i in range(len(serie)-N)])
y = serie[N:]
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = LinearRegression()
model.fit(X_train, y_train)
y_pred_all = model.predict(X)

x_all = np.arange(N, len(serie))
x_test_idx = x_all[split:]

fig, ax = plt.subplots(figsize=(12, 5))

# Zona de test (fondo gris claro)
ax.axvspan(x_test_idx[0] - 0.5, x_test_idx[-1] + 0.5,
           alpha=0.12, color='#888', label='Zona de test')

ax.plot(np.arange(len(serie)), serie * 100,
        color='#1565C0', linewidth=1.8, label='Churn real', zorder=3)
ax.plot(x_all, y_pred_all * 100,
        color='#E65100', linewidth=1.8, linestyle='--',
        label='Prediccion ventana deslizante N=3', zorder=4)

ax.axvline(x_test_idx[0] - 0.5, color='#555', linewidth=1.2,
           linestyle=':', alpha=0.8)
ax.text(x_test_idx[0] - 0.3, serie.max() * 100 * 0.97,
        'inicio test', fontsize=8, color='#555', ha='left')

tick_pos  = [0, 12, 24, 36, 47]
tick_labs = ['Ene 2020', 'Ene 2021', 'Ene 2022', 'Ene 2023', 'Dic 2023']
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labs, fontsize=9)

ax.set_xlabel('Mes', fontsize=11)
ax.set_ylabel('Tasa de churn (%)', fontsize=11)
ax.set_title('Prediccion ventana deslizante (N=3) vs churn real', fontsize=12, pad=10)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('images/B05A_fig03.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05A_fig03.png')

Guardado: images/B05A_fig03.png


---

## Cuando usar estadistica vs ML vs DL?

| Enfoque | Cuando | Limitacion |
|---------|--------|------------|
| Estadística (media movil, ARIMA) | Series cortas (<100 puntos), poca ruido, interpretabilidad requerida | No captura relaciones no lineales ni multiples series |
| ML clásico (gradient boosting con features temporales) | Series medianas (100-1000 puntos), patrones no lineales, features adicionales disponibles | Requiere ingeniería manual de features temporales |
| DL (LSTM, Transformer) | Series largas (>1000 puntos), multiples series correlacionadas, patrones complejos y jerarquicos | Alto coste computacional, requiere muchos datos para generalizar |

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')

columnas = ['Enfoque', 'Datos minimos', 'Estacionalidad', 'Coste computacional', 'Interpretabilidad']
filas = [
    ['Estadistica\n(ARIMA, media movil)', '12-50 puntos', 'Si (SARIMA)', 'Muy bajo', 'Alta'],
    ['ML clasico\n(Gradient Boosting)', '100-500 puntos', 'Con features manuales', 'Medio', 'Media'],
    ['DL\n(LSTM, Transformer)',          '>1000 puntos',  'Aprendida automaticamente', 'Alto', 'Baja'],
]

colores_filas = ['#E3F2FD', '#FFF3E0', '#E8F5E9']

tabla = ax.table(
    cellText=filas,
    colLabels=columnas,
    cellLoc='center',
    loc='center'
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(9.5)
tabla.scale(1, 2.2)

# Estilo de cabecera
for j in range(len(columnas)):
    tabla[(0, j)].set_facecolor('#1565C0')
    tabla[(0, j)].set_text_props(color='white', fontweight='bold')

# Estilo de filas de datos
for i in range(len(filas)):
    for j in range(len(columnas)):
        tabla[(i+1, j)].set_facecolor(colores_filas[i])

ax.set_title('Cuando usar cada enfoque para series temporales',
             fontsize=12, pad=16, y=0.92)
plt.tight_layout()
plt.savefig('images/B05A_fig04.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05A_fig04.png')

Guardado: images/B05A_fig04.png


---

## Ejercicio técnico

Con los datos de `data/churn_series.csv`, experimenta con ventanas de N=1, 3, 6, 12.

Preguntas guia:
- Que ventana da menor MAE en test?
- Cambia la respuesta si evaluas con RMSE en vez de MAE?
- Por que N=12 puede ser problematico con solo 48 meses de datos?

**TODO**: completar el loop de evaluación de ventanas en la celda siguiente.

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

df = pd.read_csv('data/churn_series.csv')
serie = df['churn_rate'].values

resultados = []
for N in [1, 3, 6, 12]:
    # TODO: crear X, y con ventana N
    # TODO: split 80/20 temporal
    # TODO: entrenar y predecir
    # TODO: calcular MAE y RMSE
    mae = None
    rmse = None
    resultados.append({'N': N, 'MAE': mae, 'RMSE': rmse})

print(pd.DataFrame(resultados))

    N   MAE  RMSE
0   1  None  None
1   3  None  None
2   6  None  None
3  12  None  None


In [9]:
# Solucion completa del loop de evaluacion de ventanas
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

df = pd.read_csv('data/churn_series.csv')
serie = df['churn_rate'].values

resultados = []
for N in [1, 3, 6, 12]:
    X = np.array([serie[i:i+N] for i in range(len(serie)-N)])
    y = serie[N:]
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    n_train = len(X_train)
    n_test  = len(X_test)
    resultados.append({'N': N, 'n_train': n_train, 'n_test': n_test,
                       'MAE': round(mae, 6), 'RMSE': round(rmse, 6)})

res_df = pd.DataFrame(resultados)
print('Resultados por tamano de ventana:')
print(res_df.to_string(index=False))
print()
mejor_mae  = res_df.loc[res_df['MAE'].idxmin(), 'N']
mejor_rmse = res_df.loc[res_df['RMSE'].idxmin(), 'N']
print(f'Mejor MAE:  N={mejor_mae}')
print(f'Mejor RMSE: N={mejor_rmse}')
print()
print('Conclusion: N=3 o N=6 suelen equilibrar contexto y overfitting en series de 48 meses.')
print('N=12 reduce el conjunto de entrenamiento y puede no mejorar el error en test.')

Resultados por tamano de ventana:
 N  n_train  n_test      MAE     RMSE
 1       37      10 0.006809 0.008713
 3       36       9 0.007748 0.009160
 6       33       9 0.005617 0.007272
12       28       8 0.006816 0.007678

Mejor MAE:  N=6
Mejor RMSE: N=6

Conclusion: N=3 o N=6 suelen equilibrar contexto y overfitting en series de 48 meses.
N=12 reduce el conjunto de entrenamiento y puede no mejorar el error en test.


---

## Ejercicio de decision

El equipo de operaciones tiene 36 meses de datos de tiempo de resolucion de tickets.
Proponen entrenar un LSTM para predecir el volumen de tickets del mes siguiente.

**Preguntas:**
- Es necesario un LSTM aqui?
- Que preguntarias antes de decidir?
- Hay alternativas mas simples que podrian funcionar igual o mejor?

<details>
<summary>Criterios de evaluacion (instructor)</summary>

Una respuesta madura menciona:
- Con 36 puntos, un LSTM tiene muy pocos datos: necesita cientos o miles de secuencias
  para aprender representaciones utiles. Con 36 puntos esta sobreajustando ruido.
- Una regresion lineal con ventana deslizante o gradient boosting con features de
  calendario (mes del anio, trimestre, indicador de festivos) puede ser suficiente.
- La complejidad del modelo debe justificarse con ganancia medible en una metrica
  de negocio, no con el argumento de que 'los LSTMs son mejores para series temporales'.
- Preguntas relevantes antes de decidir: hay estacionalidad clara? hay covariables
  disponibles (campanas, eventos)? cual es el coste de un error de prediccion?

Senal de comprension: el alumno rechaza el LSTM con un argumento de datos insuficientes,
no con un argumento de complejidad abstracta.

</details>

---

## Assets generados

- `data/churn_series.csv` - 48 meses de tasa de churn sintética de la empresa
- `images/B05A_fig01.png` - Serie temporal con tendencia suavizada
- `images/B05A_fig02.png` - Descomposicion: tendencia + estacionalidad + residuo
- `images/B05A_fig03.png` - Predicción ventana deslizante vs churn real
- `images/B05A_fig04.png` - Tabla comparativa: estadística vs ML vs DL